# Notebook 03 — Busca híbrida: BM25 + vetorial com RRF

Terceiro notebook do Tech Challenge 4 e o último da **base de conhecimento**.
O notebook 02 mostrou que a busca vetorial entende paráfrases, mas tem dois
limites: sempre devolve um top-k (mesmo sem evidência) e pode subestimar termos
exatos. Aqui juntamos a ela o **BM25**, a busca por palavras, e fundimos os dois
rankings com **RRF** — exatamente o desenho da IA Gen aula 02.

| Seção | Tema | Função em `src/voc_rag` | Aula de origem |
|---|---|---|---|
| 0 | Setup: base, índices e buscador | `busca.Buscador` | — |
| 1 | BM25: a busca por palavras | `busca.tokenizar_lexical`, `IndiceBM25` | PLN 03 (TF-IDF) · IA Gen 02 (slide 20) |
| 2 | Vetorial x BM25 lado a lado; reviews curtas demais | `Buscador.buscar(modo=..., min_palavras=...)` | IA Gen 02 (slide 23) |
| 3 | Fusão com RRF | `busca.rrf` | IA Gen 02 (slide 24) |
| 4 | Filtros por metadado | `busca.posicoes_permitidas` | bônus do TC4 |
| 5 | Checagem de recuperação | `avaliacao.checar_recuperacao` | IA Gen 02 (slides 26–28) |
| 6 | Sinal de recusa | `avaliacao.checar_recusa` | Requisito 4 do TC4 |

> Pré-requisitos: notebooks 01 e 02 (ou `python -m voc_rag.preparacao` e
> `python -m voc_rag.indice`).

In [1]:
import inspect
import time

import pandas as pd

from voc_rag import avaliacao, busca, config, indice
from voc_rag.preparacao import carregar_base

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)


def mostrar_codigo(funcao):
    """Imprime o código-fonte de uma função do src/, para lermos junto com o resultado."""
    print(inspect.getsource(funcao))


documentos = carregar_base()
indice_vetorial, manifesto = indice.carregar_indice()
modelo = indice.carregar_modelo()

inicio = time.perf_counter()
buscador = busca.Buscador(documentos, indice_vetorial, modelo)
print(f"{len(documentos):,} documentos | BM25 montado em {time.perf_counter() - inicio:.1f} s "
      f"| índice vetorial de {manifesto['criado_em']}")

COLUNAS = ["pontuacao", "similaridade", "pos_vetorial", "pos_bm25", "nota", "categoria", "texto"]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

42,138 documentos | BM25 montado em 0.2 s | índice vetorial de 2026-10-27T19:59:01+00:00


## Seção 1 — BM25: a busca por palavras *(PLN aula 03 · IA Gen aula 02, slide 20)*

O BM25 é o "TF-IDF com ajustes" da PLN aula 03:

- **TF** (frequência do termo na review) conta, mas **satura** (parâmetro `k1`):
  repetir "péssimo" cinco vezes não vale cinco vezes mais.
- **IDF** premia termos **raros** na base: "fiscal" pesa muito mais que "produto".
- O tamanho da review é **normalizado** (parâmetro `b`): uma review longa não ganha
  só por ter mais palavras.

Ele roda sobre o `texto_lexical` do notebook 01 (minúsculo, sem acento, contrações
expandidas). **A pergunta passa pela mesma normalização** — senão "Não" na pergunta
não casaria com "nao" na review. Observe os termos da consulta abaixo.

In [2]:
mostrar_codigo(busca.tokenizar_lexical)
mostrar_codigo(busca.IndiceBM25.buscar)

from voc_rag.texto import limpar_normalizar
pergunta = "Vc NÃO mandou a Nota Fiscal!!"
print("termos que o BM25 procura:", busca.tokenizar_lexical(limpar_normalizar(pergunta, remove_acentos=True)))

def tokenizar_lexical(texto):
    """Tokenização simples para o BM25: minúsculas + palavras alfanuméricas (IA Gen aula 02)."""
    return re.findall(r"[a-z0-9À-ÿ\-]+", texto.lower())

    def buscar(self, consulta, k=10, permitidos=None):
        """Top-k como lista de (posição, pontuação BM25). Documentos sem nenhum termo ficam de fora.

        A consulta passa pela MESMA normalização da base (sem acento, contrações
        expandidas), senão "não" na pergunta não casaria com "nao" na review.
        """
        termos = tokenizar_lexical(limpar_normalizar(consulta, remove_acentos=True))
        pontuacoes = self.bm25.get_scores(termos)
        if permitidos is not None:
            mascara = np.zeros(len(pontuacoes), dtype=bool)
            mascara[np.asarray(permitidos, dtype="int64")] = True
            pontuacoes = np.where(mascara, pontuacoes, 0.0)
        ordem = np.argsort(-pontuacoes)[:k]
        return [(int(p), float(pontuacoes[p])) for p in ordem if pontuacoes[p] > 0]

ter

In [3]:
buscador.buscar("nota fiscal", k=5, modo="bm25")[["bm25", "nota", "categoria", "texto"]]

,bm25,nota,categoria,texto
rank,,,,
1,12.14,4,informatica_acessorios,Faltou a nota fiscal
2,12.14,2,cool_stuff,Preciso da nota fiscal
3,11.51,3,perfumaria,não enviaram a nota fiscal.
4,11.51,4,cama_mesa_banho,Não veio a nota fiscal
5,11.51,1,relogios_presentes,Nem fizeram a nota fiscal


## Seção 2 — Vetorial x BM25 lado a lado *(IA Gen aula 02, slide 23)*

Duas perguntas que puxam para lados opostos:

1. **Paráfrase** — "o pedido nunca foi entregue": os clientes escrevem "não chegou",
   "ainda não recebi". A vetorial acha por sentido; o BM25 depende de as palavras
   da pergunta aparecerem na review.
2. **Termo exato** — "boleto": palavra específica, rara e decisiva.

**Observe** as colunas `pos_vetorial` e `pos_bm25`: em que posição cada review ficou
em cada busca (vazio = não apareceu entre os 50 primeiros daquela busca).

In [4]:
for pergunta in ["o pedido nunca foi entregue", "boleto"]:
    for modo in ["vetorial", "bm25"]:
        resultado = buscador.buscar(pergunta, k=5, modo=modo)
        print(f'\n"{pergunta}" — {modo.upper()} ({resultado.attrs["latencia_ms"]:.0f} ms)')
        print(resultado[["similaridade", "bm25", "nota", "texto"]].to_string())


"o pedido nunca foi entregue" — VETORIAL (14 ms)
      similaridade  bm25  nota                                                                        texto
rank                                                                                                       
1            0.845   0.0     1                                                    o pedido nao foi entregue
2            0.819   0.0     1                                                    O pedido não foi entregue
3            0.759   0.0     1                                            Meus pedidos não foram entregues.
4            0.746   0.0     5                                                Não foi entregue o meu pedido
5            0.742   0.0     1  Apesar de constar como entregue no histórico de pedidos, nada foi entregue.

"o pedido nunca foi entregue" — BM25 (29 ms)
      similaridade   bm25  nota                                                                                 texto
rank                          

### Um tropeço da busca vetorial: reviews curtas demais

Rodando com os dados reais, a pergunta "boleto" trouxe na busca vetorial reviews
como "Certinho" e "ecelente", com similaridade **0,95** — sem nenhuma relação. Com
uma pergunta de **uma palavra**, o embedding dela fica perto dos embeddings de
reviews de **uma palavra**, qualquer que seja o assunto. Cerca de 14% da base tem
1 ou 2 palavras.

**Decisão:** o `Buscador` só usa como **evidência** reviews com pelo menos
`config.MIN_PALAVRAS_EVIDENCIA` (3) palavras. Elas **continuam na base** para
contagens e temas; só não são citadas. O parâmetro `min_palavras=1` desliga o filtro
— compare abaixo.

**O que o filtro não resolve.** Nos dados reais, com o filtro ligado, o lado vetorial
de "boleto" ainda traz elogios genéricos ("Fooi bem entregue", ~0,84), e na híbrida
eles ficam intercalados com os acertos do BM25. Uma pergunta de **uma palavra** é
fraca para o embedding. Isso se resolve na etapa do RAG: a **reformulação da
pergunta** (Multi-Query, LangChain aula 03) transforma "boleto" em frases completas,
e a **checagem de evidência** descarta o que não fala do assunto.

In [5]:
for min_palavras in [1, config.MIN_PALAVRAS_EVIDENCIA]:
    resultado = buscador.buscar("boleto", k=5, modo="vetorial", min_palavras=min_palavras)
    print(f'\n"boleto" — VETORIAL com min_palavras={min_palavras}')
    print(resultado[["similaridade", "nota", "texto"]].to_string())
print(f"\nReviews com menos de {config.MIN_PALAVRAS_EVIDENCIA} palavras: "
      f"{(documentos['n_palavras'] < config.MIN_PALAVRAS_EVIDENCIA).sum():,}")


"boleto" — VETORIAL com min_palavras=1
      similaridade  nota      texto
rank                               
1            0.959     5   Certinho
2            0.954     5   ecelente
3            0.950     5     botimo
4            0.942     5  Esxelente
5            0.941     4  escelente

"boleto" — VETORIAL com min_palavras=3
      similaridade  nota                                        texto
rank                                                                 
1            0.842     5                            Fooi bem entregue
2            0.826     5                             MUITO OBRIGADA !
3            0.822     5              CONTENTE. CONTENTE COM AENTREGA
4            0.821     5  RECOMENDO. OTIMO ENTREGOU NO PRAZO PARABENS
5            0.819     1     Otima. Muito Agel com a entrega obrigado

Reviews com menos de 3 palavras: 5,983


## Seção 3 — Fusão com RRF *(IA Gen aula 02, slide 24)*

As pontuações das duas buscas estão em **escalas diferentes** (cosseno entre 0 e 1;
BM25 sem teto), então não dá para somá-las. O **Reciprocal Rank Fusion** usa só a
**posição** em cada ranking:

$$\text{RRF}(d) = \sum_{\text{rankings}} \frac{1}{k + \text{posição}(d)}, \quad k = 60$$

Quem aparece bem colocado **nas duas** buscas sobe; quem aparece em só uma ainda
entra, mais abaixo. O `k = 60` amortece a diferença entre 1º e 2º lugar.

In [6]:
mostrar_codigo(busca.rrf)

# Exemplo de brinquedo: o documento 7 é 2º num ranking e 1º no outro
print(busca.rrf([[3, 7, 9], [7, 1, 3]]))

def rrf(rankings, k=RRF_K):
    """Reciprocal Rank Fusion: funde rankings somando 1/(k + posição) (IA Gen aula 02)."""
    pontos = {}
    for ranking in rankings:
        for posicao, idx in enumerate(ranking):
            pontos[idx] = pontos.get(idx, 0.0) + 1.0 / (k + posicao + 1)
    return sorted(pontos.items(), key=lambda par: -par[1])

[(7, 0.03252247488101534), (3, 0.032266458495966696), (1, 0.016129032258064516), (9, 0.015873015873015872)]


In [7]:
pergunta = "o pedido nunca foi entregue"
resultado = buscador.buscar(pergunta, k=10, modo="hibrida")
print(f'"{pergunta}" — HÍBRIDA ({resultado.attrs["latencia_ms"]:.0f} ms)')
resultado[COLUNAS]

"o pedido nunca foi entregue" — HÍBRIDA (35 ms)


,pontuacao,similaridade,pos_vetorial,pos_bm25,nota,categoria,texto
rank,,,,,,,
1,0.0328,0.845,1,1,1,moveis_decoracao,o pedido nao foi entregue
2,0.0323,0.819,2,2,1,fashion_roupa_masculina,O pedido não foi entregue
3,0.0312,0.746,4,4,5,eletronicos,Não foi entregue o meu pedido
4,0.0303,0.724,7,5,1,bebidas,Produto não entregue. O pedido não foi entregue.
5,0.0303,0.725,6,6,2,cama_mesa_banho,Meu pedido ainda não foi. Meu pedido não foi entregue
6,0.0267,0.648,32,3,1,telefonia,Meu pedido não foi entregue De São Paulo SP foi para em Belém PA e nunca chegou aqui
7,0.0231,0.634,42,15,1,bebes,O pedido não foi entregue e consta no site como entregue.
8,0.0206,0.641,34,40,1,moveis_decoracao,Pedido não entregue
9,0.0159,0.759,3,<NA>,1,cama_mesa_banho,Meus pedidos não foram entregues.


## Seção 4 — Filtros por metadado *(bônus do TC4)*

`posicoes_permitidas` transforma filtros de negócio em posições da base, e o
`Buscador` aplica o mesmo filtro **antes** de ranquear nas duas buscas (pré-filtro,
como no notebook 02). Assim dá para responder perguntas como:

- "O que dizem os clientes **cujo pedido atrasou**?"
- "Reclamações (**nota ≤ 2**) de **móveis** no **primeiro semestre de 2018**?"

In [8]:
mostrar_codigo(busca.posicoes_permitidas)

atrasados = busca.posicoes_permitidas(documentos, entregue_com_atraso=True)
print(f"\nPedidos entregues com atraso: {len(atrasados):,} reviews")
print(buscador.buscar("o que mais incomodou na compra", k=5, permitidos=atrasados)[COLUNAS].to_string())

moveis = busca.posicoes_permitidas(documentos, nota_max=2, categorias=["moveis_decoracao"],
                                   periodo=("2018-01", "2018-06"))
print(f"\nReclamações de móveis no 1º semestre de 2018: {len(moveis):,} reviews")
buscador.buscar("problema com a montagem ou peças", k=5, permitidos=moveis)[COLUNAS + ["ano_mes"]]

def posicoes_permitidas(documentos, nota_min=None, nota_max=None, categorias=None, estados=None,
                        periodo=None, entregue_com_atraso=None):
    """Posições dos documentos que passam nos filtros (None = sem filtro naquele campo).

    `periodo` é um par ("2018-01", "2018-06") de ano_mes, inclusivo.
    """
    filtro = pd.Series(True, index=documentos.index)
    if nota_min is not None:
        filtro &= documentos["nota"] >= nota_min
    if nota_max is not None:
        filtro &= documentos["nota"] <= nota_max
    if categorias is not None:
        filtro &= documentos["categoria"].isin(categorias)
    if estados is not None:
        filtro &= documentos["estado"].isin(estados)
    if periodo is not None:
        filtro &= documentos["ano_mes"].between(*periodo)
    if entregue_com_atraso is not None:
        filtro &= documentos["entregue_com_atraso"].fillna(not entregue_com_atraso) == entregue_com_atraso
    return np.flatnonzero(filtro.to_numpy())


Pedidos ent

      pontuacao  similaridade  pos_vetorial  pos_bm25  nota           categoria                                                                                                                                                                                     texto
rank                                                                                                                                                                                                                                                                     
1        0.0292         0.610             9         8     1     cama_mesa_banho                                                                                                            já comprei por diversas vezes nessa loja e esse foi a compra que mais atrasou.
2        0.0277         0.630             7        18     5    malas_acessorios                                                                                                                 poderiam d

,pontuacao,similaridade,pos_vetorial,pos_bm25,nota,categoria,texto,ano_mes
rank,,,,,,,,
1,0.0308,0.494,8,2,1,moveis_decoracao,"O produto veio incompleto, sem condições de montagem e uso",2018-03
2,0.0287,0.436,15,5,2,moveis_decoracao,"Tamanho. Faltou tamanho na descrição, acho q veio faltando peças, pois é muito baixa",2018-06
3,0.0283,0.547,2,22,1,moveis_decoracao,"Quadro quebrado. Infelizmente o meu quadro veio com a moldura quebrada, danificando a ...",2018-06
4,0.0282,0.570,1,25,2,moveis_decoracao,o produto chegou e como estava em viagem demorei uma semana para montar e quando fui m...,2018-03
5,0.0276,0.446,13,12,1,moveis_decoracao,"Luminárias. O produto veio faltando uma unidade, e ainda não consegui entrar em contat...",2018-06


## Seção 5 — Checagem de recuperação *(IA Gen aula 02, slides 26–28)*

Na aula 02, o gabarito era "pergunta → o documento certo". Aqui uma pergunta como
"o pedido não chegou" tem **centenas** de reviews relevantes, então medimos a
**precisão@10**: das 10 reviews devolvidas, quantas falam do tema?

O "juiz" é uma expressão regular por pergunta (`avaliacao.PERGUNTAS_CHECAGEM`).
**Limite:** um juiz por palavras tende a **favorecer o BM25**. Para compensar, as
perguntas usam palavras diferentes das do padrão — e você deve ler o top-k de olho.
Esta é uma checagem de sanidade da base de conhecimento; a avaliação completa
(gold set com julgamento humano) vem no protocolo de avaliação do RAG.

In [9]:
checagem = avaliacao.checar_recuperacao(buscador, k=10)
tabela = checagem.pivot(index="pergunta", columns="modo", values="precisao@10")[["vetorial", "bm25", "hibrida"]]
tabela.loc["MÉDIA"] = tabela.mean()
print("Precisão@10 (fração do top-10 que fala do tema):")
print(tabela.round(2).to_string())
print("\nLatência média (ms):", checagem.groupby("modo")["latencia_ms"].mean().round(0).to_dict())

Precisão@10 (fração do top-10 que fala do tema):
modo                                         vetorial  bm25  hibrida
pergunta                                                            
comprei mais de um item e só veio uma parte      0.50  0.70     0.40
embalagem chegou toda amassada                   0.90  1.00     0.90
entrega mais rápida do que o esperado            1.00  1.00     1.00
material de ótima qualidade, recomendo           1.00  1.00     1.00
o pedido nunca foi entregue                      0.80  0.70     0.80
problema com a nota fiscal da compra             0.90  1.00     1.00
produto chegou com defeito ou danificado         1.00  1.00     1.00
quero meu dinheiro de volta                      0.90  1.00     1.00
recebi um produto diferente do anunciado         0.90  1.00     1.00
vendedor não responde as mensagens               0.60  0.70     0.60
MÉDIA                                            0.85  0.91     0.87

Latência média (ms): {'bm25': 36.0, 'hibrida': 39.0, 

**Como ler:** a híbrida não precisa vencer em toda linha — o objetivo é ela não
perder feio em nenhuma, porque herda o melhor de cada busca. Repare nas linhas em
que vetorial e BM25 discordam muito: são as que justificam a fusão.

## Seção 6 — Sinal de recusa *(requisito 4 do TC4)*

O TC4 exige **recusar** quando não há evidência. A busca sempre devolve um top-k,
então precisamos de um sinal: a **maior similaridade** entre a pergunta e as
reviews recuperadas. Perguntas do tema ficam bem acima de perguntas fora dele.

`config.LIMIAR_EVIDENCIA` (0,55) é **provisório**, tirado do teste da Etapa 3. Ele
será calibrado no protocolo de avaliação; o pipeline RAG usará este número, junto
com a checagem de evidência pelo LLM, para decidir entre responder e recusar.

In [10]:
recusa = avaliacao.checar_recusa(buscador)
print(recusa.to_string(index=False))
dentro = recusa.loc[recusa["grupo"] == "dentro", "similaridade_top1"]
fora = recusa.loc[recusa["grupo"] == "fora", "similaridade_top1"]
print(f"\nmenor similaridade DENTRO do tema: {dentro.min():.3f} | maior FORA do tema: {fora.max():.3f} "
      f"| limiar atual: {config.LIMIAR_EVIDENCIA}")

 grupo                                    pergunta  similaridade_top1  tem_evidencia
dentro                 o pedido nunca foi entregue              0.845           True
dentro    produto chegou com defeito ou danificado              0.921           True
dentro       entrega mais rápida do que o esperado              0.874           True
dentro    recebi um produto diferente do anunciado              0.997           True
dentro        problema com a nota fiscal da compra              0.818           True
dentro          vendedor não responde as mensagens              0.732           True
dentro comprei mais de um item e só veio uma parte              0.826           True
dentro                 quero meu dinheiro de volta              0.997           True
dentro              embalagem chegou toda amassada              0.795           True
dentro      material de ótima qualidade, recomendo              0.919           True
  fora                 qual é a capital da França?              0

## Resumo das decisões

| Decisão | Por quê |
|---|---|
| BM25 sobre `texto_lexical`, consulta com a mesma normalização | "não" e "nao" precisam casar; acentos variam entre clientes |
| BM25 reconstruído a cada carga, não gravado | ~42 mil reviews curtas: poucos segundos, sem arquivo para sair de sincronia |
| RRF com k = 60 sobre os 50 primeiros de cada busca | usa posições, não pontuações em escalas diferentes; valor da aula 02 |
| Reviews com menos de 3 palavras não viram evidência | com pergunta curta, a vetorial as puxa com similaridade alta e sem sentido |
| Mesmo pré-filtro de metadado nas duas buscas | o filtro vale para a resposta inteira, não para uma das buscas |
| Precisão@10 com juiz por regex | sem gabarito único por pergunta; checagem de sanidade, não avaliação final |
| Similaridade do top-1 como sinal de recusa | a busca sempre devolve algo; limiar provisório a calibrar |

**A base de conhecimento está pronta:** documentos com metadados (notebook 01),
índice vetorial (02) e busca híbrida com filtros (03). O próximo passo do TC4 é o
**pipeline RAG**: transformar a pergunta, recuperar com o `Buscador`, checar a
evidência e gerar a resposta citando os `review_id` — ou recusar.